선형 회귀 모델LinearRegrssion

사이킷런 선형모델 선형회귀모델

model <=

fit(X,Y) -> 회귀선

model predict -> 특정 X에 대한 Y값을 예측

회귀모델은 추세trend와 계절성을 사람이 변수로 만들어서 입력해야 한다.

시계열 전용 모델이 있는데 시계열 전용 모델은 최근 값과 최근 오차가 다음 값에 어떻게 이어지는가를 데이터에서 스스로 찾는다(학습한다).

- SARIMA
- 지수평활exponential smoothing




## SARIMA
ARIMA, AR(자기회귀autoregressive), I(누적integrated), MA(이동평균 오차moving average)

자기회귀는 지난 달 값이 높았으면 이번 달도 높다처럼 과거 값으로 현재를 설명하는 것이고 누적은 값 대신 전월과의 차이(차분)를 모델링해서 추세를 없애고 이동평균은 지난 달 예측이 빗나간만큼 이번달 값을 보정하겠다.

# 모델의 평가 설계, 개선
모델이 예측한 결과를 얼마나 믿을 수 있는지를 따져볼 수 있는 도구가 필요하다. 이때 RMSE, ME, MASE 같은 평가 지표를 더하고 예측기간별로 오차를 나누어보는 롤링 원점 평가를 설계하고 로그변화 회귀, 계절시차 특성, 학습기간선택, 예측/결합 등의 방법으로 모델을 다듬을 수 있다.

## 지표
- RMSE root mean squared error평균제곱근 오차: 큰 오차가 얼마나 있는가?
- ME mean error편향: 한쪽으로 치우쳐져서 예측하는가?
- MASE mean absolute scaled error: 계절 나이브보다 얼마나 나은가?

### RMSE
RMSE는 오차를 제곱해서 평균을 구한 다음에 다시 제곱근을 씌운 값을 의미한다. 제곱을 하게되면 큰 오차가 더 커지게 된다. 그래서 RMSE를 구하게 되면 큰 오차에 민감하게 반응할 수 있게 된다.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
plt.rcParams['font.family'] = 'Malgun Gothic'

def load_dataset(filename):
    BASE_DIR = './data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding='euc-kr')

In [4]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
# RMSE, MAE: 늘 조금씩 틀리는 예측, 가끔 크게 틀리는 예측

actual = np.full(12, 2_000_000)
steady = actual+ np.array([50_000, -50_000]*6)
spiky = actual.copy()
spiky[6] = actual[6] + 550_000
spiky = spiky +np.array([1_000]*6 + [0] + [1_000]*5)

errors = actual - spiky
print(f'MAE = {round(np.abs(errors).mean())}') #평균의 오류
#RMSE
#sqrt()
print(f'RMSE = {round(np.sqrt((errors ** 2).mean()))}')#잘못된 게 확 드러남, 큰 거에 제곱하면 뻥뛰기됨


MAE = 46750
RMSE = 158774


In [6]:
for name, pred in [('늘 조금씩 틀림', steady), ('가끔 크게 틀림', spiky)]:
    mae = mean_absolute_error(actual, pred) # 모델이 예측한 값
    rmse = root_mean_squared_error(actual, pred)
    print(f'{name} 모델의 MAE = {mae:,.0f} RMSE = {rmse:,.0f}')

늘 조금씩 틀림 모델의 MAE = 50,000 RMSE = 50,000
가끔 크게 틀림 모델의 MAE = 46,750 RMSE = 158,774


## 편향 bias
